In [1]:
import pandas as pd
import numpy as np
import os

# Đọc file dữ liệu hoàn toàn bằng số đã xử lý xong ở file 04
df = pd.read_csv('../data/processed/clean_data.csv')

print("THÔNG TIN BỘ DỮ LIỆU SỐ BAN ĐẦU:")
print(f"- Số lượng dòng: {df.shape[0]}")
print(f"- Số lượng cột: {df.shape[1]}")
print("\n Kiểm tra vài dòng đầu:")
df.head(5)

THÔNG TIN BỘ DỮ LIỆU SỐ BAN ĐẦU:
- Số lượng dòng: 2369
- Số lượng cột: 15

 Kiểm tra vài dòng đầu:


,price_raw,area_raw,bathrooms,floors,address_encoded,interior_encoded,legal_encoded,dir_Bắc,dir_Nam,dir_Tây,dir_Tây Bắc,dir_Tây Nam,dir_Đông,dir_Đông Bắc,dir_Đông Nam
0,5.95,48.0,2.0,2.0,14,1,2,0,0,0,0,0,0,0,1
1,16.90,90.0,12.0,3.0,18,4,2,0,0,0,0,0,0,0,0
2,22.00,78.0,4.0,5.0,15,4,2,0,0,0,0,0,0,0,0
3,6.80,40.5,3.0,3.0,20,4,2,0,0,0,0,0,0,0,0
4,7.25,36.8,5.0,5.0,17,4,2,0,0,0,0,0,0,0,0


In [2]:
from sklearn.model_selection import train_test_split


X_raw = df.drop(columns=['price_raw'])
y_raw = df['price_raw']

# Lần 1: Tách 20% làm tập Test. 80% còn lại làm tập tạm (Train + Val)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_raw, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['dir_Bắc'] 
)

# Lần 2: Tách 80% tập tạm thành Train (60% tổng) và Validation (20% tổng)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=0.25, 
    random_state=42, 
    stratify=X_temp['dir_Bắc']
)

print(" KÍCH THƯỚC CÁC TẬP DỮ LIỆU SAU KHI CHIA:")
print(f"- Tập Train      (60%): {X_train.shape[0]} dòng")
print(f"- Tập Validation (20%): {X_val.shape[0]} dòng")
print(f"- Tập Test       (20%): {X_test.shape[0]} dòng")

 KÍCH THƯỚC CÁC TẬP DỮ LIỆU SAU KHI CHIA:
- Tập Train      (60%): 1421 dòng
- Tập Validation (20%): 474 dòng
- Tập Test       (20%): 474 dòng


In [3]:

# 1. Biến đổi log cho biến mục tiêu y (Giá)
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

# 2. Biến đổi log cho biến diện tích (Area) trên ma trận đặc trưng
X_train['area_log'] = np.log1p(X_train['area_raw'])
X_val['area_log'] = np.log1p(X_val['area_raw'])
X_test['area_log'] = np.log1p(X_test['area_raw'])

# 3. Xóa cột diện tích thô cũ
X_train = X_train.drop(columns=['area_raw'])
X_val = X_val.drop(columns=['area_raw'])
X_test = X_test.drop(columns=['area_raw'])

print(" THỐNG KÊ GIÁ TRỊ CỰC ĐẠI (MAX) SAU KHI LOG TRANSFORM:")
print(f"- Train Max Log Price: {y_train_log.max():.4f} | Train Max Log Area: {X_train['area_log'].max():.4f}")
print(f"- Val Max Log Price:   {y_val_log.max():.4f} | Val Max Log Area:   {X_val['area_log'].max():.4f}")

 THỐNG KÊ GIÁ TRỊ CỰC ĐẠI (MAX) SAU KHI LOG TRANSFORM:
- Train Max Log Price: 5.7071 | Train Max Log Area: 6.3869
- Val Max Log Price:   5.1417 | Val Max Log Area:   6.3469


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled_arr = scaler.fit_transform(X_train)

X_val_scaled_arr = scaler.transform(X_val)
X_test_scaled_arr = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled_arr, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled_arr, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled_arr, columns=X_train.columns)

print("KIỂM TRA ĐỘ CHUẨN HÓA (Kiểm tra Mean và Max của vài dòng đầu):")
print(X_train_scaled.describe().loc[['min', 'max', 'mean']].iloc[:, :4])

KIỂM TRA ĐỘ CHUẨN HÓA (Kiểm tra Mean và Max của vài dòng đầu):
         bathrooms        floors  address_encoded  interior_encoded
min  -1.583184e+00 -1.394845e+00    -3.204090e+00     -2.919998e+00
max   5.749963e+00  4.099471e+00     1.275818e+00      1.913783e+00
mean  1.125068e-16  9.500571e-17     2.437647e-17      2.237635e-16


In [5]:

output_dir = '../data/processed'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

train_final = X_train_scaled.copy()
train_final['target_price_log'] = y_train_log.values

val_final = X_val_scaled.copy()
val_final['target_price_log'] = y_val_log.values

test_final = X_test_scaled.copy()
test_final['target_price_log'] = y_test_log.values

train_final.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
val_final.to_csv(os.path.join(output_dir, 'validation.csv'), index=False)
test_final.to_csv(os.path.join(output_dir, 'test.csv'), index=False)

print(f" ĐÃ XUẤT THÀNH CÔNG 3 TẬP SỐ VÀO THƯ MỤC '{output_dir}/':")
print(f"- {os.path.join(output_dir, 'train.csv')}")
print(f"- {os.path.join(output_dir, 'validation.csv')}")
print(f"- {os.path.join(output_dir, 'test.csv')}")

 ĐÃ XUẤT THÀNH CÔNG 3 TẬP SỐ VÀO THƯ MỤC '../data/processed/':
- ../data/processed\train.csv
- ../data/processed\validation.csv
- ../data/processed\test.csv


Lưu giá trị chuẩn hóa

In [6]:
import joblib

# Tạo thư mục models nếu chưa có
model_dir = os.path.join('..', 'models')
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Lưu scaler
scaler_path = os.path.join(model_dir, 'standard_scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f" ĐÃ LƯU SCALER THÀNH CÔNG TẠI: {scaler_path}")

 ĐÃ LƯU SCALER THÀNH CÔNG TẠI: ..\models\standard_scaler.pkl
